# Machine Learning Zoomcamp 2026 - Homework 3: Classification

This notebook explores a lead-scoring dataset, prepares a reproducible train/validation/test split, and evaluates logistic regression models. All preprocessing that learns from data is fitted using the training set.

## 1. Data loading and preparation

Load the course dataset and fill missing feature values with `NA` for categorical columns and `0.0` for numerical columns. The target is kept separate from feature imputation.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, mutual_info_score
from sklearn.model_selection import train_test_split

DATA_URL = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
TARGET = "converted"

df = pd.read_csv(DATA_URL)
if TARGET not in df.columns:
    raise ValueError(f"Expected target column '{TARGET}' in the dataset.")

X = df.drop(columns=TARGET).copy()
y = df[TARGET].copy()

categorical_features = X.select_dtypes(
    include=["object", "category", "string"]
).columns.tolist()
numerical_features = X.select_dtypes(include="number").columns.tolist()

X.loc[:, categorical_features] = X[categorical_features].fillna("NA")
X.loc[:, numerical_features] = X[numerical_features].fillna(0.0)

print(f"Dataset shape: {df.shape}")
print(f"Number of categorical features: {len(categorical_features)}")
print(f"Number of numerical features: {len(numerical_features)}")

## 2. Q1 and Q2: feature exploration

Q1 finds the most frequent industry. Q2 computes correlations among numerical features (excluding the target) and reports the pair with the greatest correlation coefficient.

In [ ]:
if "industry" not in X.columns:
    raise ValueError("Expected feature column 'industry' in the dataset.")

industry_mode = X["industry"].mode().iloc[0]
print(f"Q1 - Most frequent industry: {industry_mode}")

correlation_matrix = X[numerical_features].corr()
print("\nQ2 - Correlation matrix for numerical features:")
print(correlation_matrix)

upper_triangle = correlation_matrix.where(
    np.triu(np.ones(correlation_matrix.shape, dtype=bool), k=1)
)
correlation_pairs = upper_triangle.stack()
if correlation_pairs.empty:
    raise ValueError("At least two numerical features are required to compare correlations.")

highest_pair = correlation_pairs.idxmax()
highest_correlation = correlation_pairs.loc[highest_pair]
print(
    f"Highest-correlated pair: {highest_pair[0]} and {highest_pair[1]} "
    f"(correlation: {highest_correlation:.4f})"
)

## 3. Dataset split

Reserve 20% for testing, then split the remaining data into 75% training and 25% validation. This produces a 60% / 20% / 20% split using the requested random seed.

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42
)

print(f"Training examples: {len(X_train)} ({len(X_train) / len(X):.0%})")
print(f"Validation examples: {len(X_val)} ({len(X_val) / len(X):.0%})")
print(f"Test examples: {len(X_test)} ({len(X_test) / len(X):.0%})")

## 4. Q3: mutual information

Calculate mutual information between each categorical feature and the conversion target using training rows only. Scores are rounded to two decimal places.

In [ ]:
mutual_information = pd.Series(
    {
        feature: mutual_info_score(X_train[feature], y_train)
        for feature in categorical_features
    },
    name="mutual_information",
).sort_values(ascending=False)

if mutual_information.empty:
    raise ValueError("No categorical features are available for mutual information analysis.")

print("Q3 - Mutual information scores:")
print(mutual_information.round(2))
print(
    f"Highest-scoring categorical feature: {mutual_information.index[0]} "
    f"(score: {mutual_information.iloc[0]:.2f})"
)

## 5. Q4: logistic regression baseline

Fit a dense one-hot encoder on the training set and train logistic regression with the specified parameters. The same fitted encoder transforms validation data to keep feature columns aligned.

In [ ]:
def records_without(features, excluded_feature=None):
    selected_features = [
        feature for feature in features.columns if feature != excluded_feature
    ]
    return features[selected_features].to_dict(orient="records")


train_records = records_without(X_train)
val_records = records_without(X_val)

vectorizer = DictVectorizer(sparse=False)
X_train_encoded = vectorizer.fit_transform(train_records)
X_val_encoded = vectorizer.transform(val_records)

baseline_model = LogisticRegression(
    solver="liblinear", C=1.0, max_iter=1000, random_state=42
)
baseline_model.fit(X_train_encoded, y_train)
baseline_accuracy = accuracy_score(y_val, baseline_model.predict(X_val_encoded))

print(f"Q4 - Validation accuracy: {baseline_accuracy:.2f}")

## 6. Q5: feature elimination

Exclude each requested feature separately. For every experiment, fit a fresh one-hot encoder and logistic regression model, then compare its validation accuracy with the unrounded baseline accuracy.

In [ ]:
features_to_exclude = [
    "lead_source",
    "number_of_courses_viewed",
    "interaction_count",
]
missing_features = [feature for feature in features_to_exclude if feature not in X_train.columns]
if missing_features:
    raise ValueError(f"Expected features are missing from the dataset: {missing_features}")

accuracy_differences = {}
for excluded_feature in features_to_exclude:
    reduced_train_records = records_without(X_train, excluded_feature)
    reduced_val_records = records_without(X_val, excluded_feature)

    reduced_vectorizer = DictVectorizer(sparse=False)
    reduced_X_train = reduced_vectorizer.fit_transform(reduced_train_records)
    reduced_X_val = reduced_vectorizer.transform(reduced_val_records)

    reduced_model = LogisticRegression(
        solver="liblinear", C=1.0, max_iter=1000, random_state=42
    )
    reduced_model.fit(reduced_X_train, y_train)
    reduced_accuracy = accuracy_score(y_val, reduced_model.predict(reduced_X_val))
    accuracy_differences[excluded_feature] = baseline_accuracy - reduced_accuracy
    print(
        f"Accuracy difference without {excluded_feature}: "
        f"{accuracy_differences[excluded_feature]:.6f}"
    )

least_impactful_feature = min(accuracy_differences, key=accuracy_differences.get)
print(f"Q5 - Feature with the smallest accuracy difference: {least_impactful_feature}")

## 7. Q6: regularization tuning

Train models with all features for the requested `C` values. Report each validation accuracy to three decimal places; when accuracies tie, choose the smallest `C`.

In [ ]:
regularization_values = [0.000001, 0.00001, 0.0001, 0.001]
regularization_accuracies = {}

for c_value in regularization_values:
    model = LogisticRegression(
        solver="liblinear", C=c_value, max_iter=1000, random_state=42
    )
    model.fit(X_train_encoded, y_train)
    accuracy = accuracy_score(y_val, model.predict(X_val_encoded))
    regularization_accuracies[c_value] = accuracy
    print(f"C={c_value:g} - Validation accuracy: {accuracy:.3f}")

best_c = min(
    regularization_accuracies,
    key=lambda value: (-regularization_accuracies[value], value),
)
print(f"Q6 - Best C: {best_c:g} (accuracy: {regularization_accuracies[best_c]:.3f})")